In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-03-28 06:38:51.003811: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774679931.210510      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774679931.271939      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774679931.772992      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774679931.773036      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774679931.773039      24 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train.npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train.npy'))
X_val = np.load(os.path.join(INPUT_PATH, 'X_val.npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val.npy'))
X_test = np.load(os.path.join(INPUT_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test.npy'))

In [4]:
import numpy as np

X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (36597, 10, 133) float32
y_val:   (36597,) int32
X_test:  (45552, 10, 133) float32
y_test:  (45552,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [33804  2793]
Test class distribution: [42337  3215]


In [5]:
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional

def create_model(n_lstm_units, dropout_rate):
    model = Sequential([
        Bidirectional(
            LSTM(n_lstm_units, return_sequences=True),
            input_shape=(X_train.shape[1], X_train.shape[2])
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Bidirectional(
            LSTM(max(n_lstm_units // 2, 8), return_sequences=False)
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )
    return model

print("✅ Đã khởi tạo create_model")

✅ Đã khởi tạo create_model


In [6]:
from sklearn.utils.class_weight import compute_class_weight
import tensorflow.keras.backend as K
import numpy as np
import tensorflow as tf

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}
print("Trọng số lớp Train:", class_weights_dict)

lstm_units_options = [16, 32, 50]
dropout_rate_options = [0.3, 0.4, 0.5]
batch_size_options = [256, 512]
num_epochs = 15

results = []

for n_units in lstm_units_options:
    for dropout_rate in dropout_rate_options:
        for batch_size in batch_size_options:
            print(f"\nĐang test: units={n_units}, dropout={dropout_rate}, batch_size={batch_size}")

            K.clear_session()

            model = create_model(n_lstm_units=n_units, dropout_rate=dropout_rate)

            history = model.fit(
                X_train, y_train,
                epochs=num_epochs,
                batch_size=batch_size,
                class_weight=class_weights_dict,
                validation_data=(X_val, y_val),
                shuffle=True,
                verbose=0
            )

            val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision = model.evaluate(
                X_val, y_val, verbose=0
            )

            print(
                f"Val AUROC: {val_auroc:.4f} | "
                f"Val AUPRC: {val_auprc:.4f} | "
                f"Recall: {val_recall:.4f} | "
                f"Precision: {val_precision:.4f}"
            )

            results.append((
                n_units, dropout_rate, batch_size,
                val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision
            ))

best_result = sorted(results, key=lambda x: x[5], reverse=True)[0]

print("\n" + "="*60)
print(f"Best Hyperparameters:")
print(f"- LSTM Units:   {best_result[0]}")
print(f"- Dropout Rate: {best_result[1]}")
print(f"- Batch Size:   {best_result[2]}")
print(f"- Val AUROC:    {best_result[5]:.4f}")
print(f"- Val AUPRC:    {best_result[6]:.4f}")
print("="*60)

Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}

Đang test: units=16, dropout=0.3, batch_size=256


I0000 00:00:1774679970.669392      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774679970.675362      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1774679978.875904      73 cuda_dnn.cc:529] Loaded cuDNN version 91002


Val AUROC: 0.7925 | Val AUPRC: 0.3223 | Recall: 0.5532 | Precision: 0.2922

Đang test: units=16, dropout=0.3, batch_size=512
Val AUROC: 0.7887 | Val AUPRC: 0.3188 | Recall: 0.5582 | Precision: 0.2799

Đang test: units=16, dropout=0.4, batch_size=256
Val AUROC: 0.7927 | Val AUPRC: 0.3101 | Recall: 0.5510 | Precision: 0.2996

Đang test: units=16, dropout=0.4, batch_size=512
Val AUROC: 0.7941 | Val AUPRC: 0.3449 | Recall: 0.6040 | Precision: 0.2772

Đang test: units=16, dropout=0.5, batch_size=256
Val AUROC: 0.8017 | Val AUPRC: 0.3311 | Recall: 0.6144 | Precision: 0.2828

Đang test: units=16, dropout=0.5, batch_size=512
Val AUROC: 0.8054 | Val AUPRC: 0.3283 | Recall: 0.6047 | Precision: 0.2946

Đang test: units=32, dropout=0.3, batch_size=256
Val AUROC: 0.7601 | Val AUPRC: 0.3080 | Recall: 0.4758 | Precision: 0.3279

Đang test: units=32, dropout=0.3, batch_size=512
Val AUROC: 0.7721 | Val AUPRC: 0.3230 | Recall: 0.5048 | Precision: 0.3317

Đang test: units=32, dropout=0.4, batch_size=256


In [7]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K

K.clear_session()

final_model = create_model(
    n_lstm_units=best_result[0],
    dropout_rate=best_result[1]
)

early_stopping = EarlyStopping(
    monitor='val_auroc',
    mode='max',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_auroc',
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_lstm_model.keras',
    monitor='val_auroc',
    mode='max',
    save_best_only=True,
    verbose=1
)

print("🚀 ĐANG HUẤN LUYỆN FINAL MODEL...")
history = final_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=best_result[2],
    class_weight=class_weights_dict,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    shuffle=True,
    verbose=1
)

🚀 ĐANG HUẤN LUYỆN FINAL MODEL...
Epoch 1/50
282/285 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5330 - auprc: 0.1360 - auroc: 0.6486 - loss: 0.6790 - precision: 0.1072 - recall: 0.6863
Epoch 1: val_auroc improved from -inf to 0.82287, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - accuracy: 0.5345 - auprc: 0.1367 - auroc: 0.6496 - loss: 0.6782 - precision: 0.1076 - recall: 0.6863 - val_accuracy: 0.7937 - val_auprc: 0.2906 - val_auroc: 0.8229 - val_loss: 0.4749 - val_precision: 0.2222 - val_recall: 0.6813 - learning_rate: 0.0010
Epoch 2/50
284/285 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7511 - auprc: 0.2883 - auroc: 0.8186 - loss: 0.5324 - precision: 0.1997 - recall: 0.7504
Epoch 2: val_auroc improved from 0.82287 to 0.84437, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7512 - auprc: 0.2883 - auroc: 0.8187 - loss: 0.5323 - precision: 0.1997 - recall: 0.7506 - val_accuracy: 0.7829 - val_auprc

In [8]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print("⏳ Đang chấm điểm trên tập Test...")
test_metrics = final_model.evaluate(X_test, y_test, verbose=0)

print("KẾT QUẢ TRÊN TẬP TEST:")
print(f"Test Loss:      {test_metrics[0]:.4f}")
print(f"Test Accuracy:  {test_metrics[1]:.4f}")
print(f"Test AUROC:     {test_metrics[2]:.4f}")
print(f"Test AUPRC:     {test_metrics[3]:.4f}")
print(f"Test Recall:    {test_metrics[4]:.4f}")
print(f"Test Precision: {test_metrics[5]:.4f}")

print("📦 Đang dự đoán trên X_test...")
pred_prob = final_model.predict(X_test, verbose=0).flatten()
pred_label = (pred_prob >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, pred_label))

print("\nClassification Report:")
print(classification_report(y_test, pred_label, digits=4))

print(f"ROC-AUC (sklearn): {roc_auc_score(y_test, pred_prob):.4f}")
print(f"PR-AUC  (sklearn): {average_precision_score(y_test, pred_prob):.4f}")

df_lstm = pd.DataFrame({
    'y_true': y_test,
    'pred_lstm': pred_prob,
    'pred_label': pred_label
})
df_lstm.to_csv('predictions_lstm.csv', index=False)

print("✅ Đã xuất 'predictions_lstm.csv'")

⏳ Đang chấm điểm trên tập Test...
KẾT QUẢ TRÊN TẬP TEST:
Test Loss:      0.4015
Test Accuracy:  0.8113
Test AUROC:     0.8692
Test AUPRC:     0.3591
Test Recall:    0.7736
Test Precision: 0.2402
📦 Đang dự đoán trên X_test...
Confusion Matrix:
[[34470  7867]
 [  728  2487]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9793    0.8142    0.8891     42337
           1     0.2402    0.7736    0.3666      3215

    accuracy                         0.8113     45552
   macro avg     0.6098    0.7939    0.6279     45552
weighted avg     0.9272    0.8113    0.8523     45552

ROC-AUC (sklearn): 0.8694
PR-AUC  (sklearn): 0.3591
✅ Đã xuất 'predictions_lstm.csv'
